# 🏗️ NEXUS ENSAIO: Sovereign Cloud Renderer

Este notebook executa o pipeline industrial de renderização do Nexus. Ele utiliza a arquitetura baseada em Rust (WGPU/WGSL) para simulação paralela de partículas na GPU fornecida pelo Google Colab, mantendo a orquestração via Python.

**Regra de Ouro SOTA:** O Python orquestra a lógica semântica (`director.py`), o Rust renderiza a física massiva (`render_native`). Nunca os misture na mesma camada.

### Fase 1: Inicialização e Montagem do Drive
Conecta o notebook ao barramento central de dados do projeto.

In [ ]:
from google.colab import drive
import os
import json
import sys

# 1. Montar o Google Drive
drive.mount('/content/drive')

# 2. Configuração de Caminhos Globais
DRIVE_PATH = '/content/drive/MyDrive/nexus_pipeline'
PIPELINE_PATH = f'{DRIVE_PATH}/staging/ensaio/pipeline'

# Adiciona o pipeline ao sys.path para conseguirmos importar os scripts
sys.path.append(PIPELINE_PATH)

print("✅ Drive montado e rotas configuradas.")

### Fase 2: Setup Industrial (Headless Environment)
Prepara o sistema Linux do Colab instalando dependências críticas de sistema (Vulkan, FFmpeg) e o compilador Rust, garantindo um ambiente puro para alta performance gráfica.

In [ ]:
import importlib
import colab_render

# Força o recarregamento do módulo caso ele tenha sido alterado no Drive
importlib.reload(colab_render)

# Prepara as bibliotecas C e Rust
colab_render.setup_colab_environment(DRIVE_PATH)
print("✅ Ambiente SOTA inicializado com sucesso.")

### Fase 3: Adição de Gatilhos Semânticos (Opcional)
Utilize esta célula caso queira adicionar novas palavras-chave de ativação visual ao dicionário sem precisar rodar scripts locais. O script cuidará de não duplicar registros.

In [ ]:
EVENTS_MAP_PATH = f'{PIPELINE_PATH}/events_map.json'

# Novos gatilhos que desejamos mapear no documentário
new_events = [
    {"trigger_word": "cold", "vtype": "icon", "asset": "icon_biohazard.png"},
    {"trigger_word": "war", "vtype": "vector_map", "asset": "world.json"}
]

try:
    # Carregar existentes
    with open(EVENTS_MAP_PATH, 'r') as f: 
        existing_events = json.load(f)
        
    # Evitar duplicados simples pela palavra-chave
    current_triggers = [e['trigger_word'].lower() for e in existing_events]
    
    for ne in new_events:
        if ne['trigger_word'].lower() not in current_triggers:
            existing_events.append(ne)
            print(f"Adicionado novo gatilho: '{ne['trigger_word']}' -> {ne['asset']}")
            
    # Salvar de volta no Drive
    with open(EVENTS_MAP_PATH, 'w') as f: 
        json.dump(existing_events, f, indent=4)
        
    print("✅ Dicionário Semântico Atualizado e Consolidado.")
except FileNotFoundError:
    print(f"⚠️ Arquivo events_map.json não encontrado em {EVENTS_MAP_PATH}. Verifique se o sync_drive local rodou corretamente.")

### Fase 4: Orquestração e Renderização Absoluta
Aqui a verdadeira engenharia entra em ação. 
1. O **Diretor** consolida o Áudio + WhisperX + Events Map para gerar a partitura (`narrative.json`) com limite de lookback e tolerância de 15s de exposição.
2. O **Renderizador Nativo** injeta as variáveis Vulkan Headless e aciona a engine massiva em Rust para gerar o .mp4 usando aceleração por GPU.

In [ ]:
# 1. Geração da Partitura Visual (Narrative.json)
# Lê o script.wav e o words.json do Drive gerados pelo WhisperX na etapa de áudio
print("\n--- 🧠 FASE 1: Nexus Director ---")
colab_render.run_director(DRIVE_PATH)

# 2. Renderização de Alta Fidelidade (Rust/WGSL)
# O Output será salvo diretamente em: /audio_ready/ensaio/cold_war_cloud_final.mp4
print("\n--- 🎬 FASE 2: Sovereign Render Engine ---")
colab_render.run_render(DRIVE_PATH)

print("\n🚀 PROCESSO CONCLUÍDO. O VÍDEO ESTÁ NO SEU GOOGLE DRIVE.")